<a href="https://colab.research.google.com/github/RADHESHYAM07/AI_powered_LinkedIN_Scraper/blob/main/AI_Powred_linkedIn_Scrapper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [36]:
!pip install gradio #install gradio for webapp

In [37]:
!pip install flair #flair for ai Skill detection


In [49]:
import gradio as gr #Gradio to build webapp
import requests  # to fetch the web Pages
from bs4 import BeautifulSoup # this library is to parse HTML
import pandas as pd # to handle datatables
from datetime import datetime
import time # for adding delays in requests


# import flair tools for AI skill detection
from flair.models import SequenceTagger
from flair.data import Sentence

import threading # for smooth background Tasks
import os #to save File
import random  # for random Delays


# import retry tools to handle web error
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

### Load AI MODEL

In [50]:
# load Flair's Skill detection MOdel
flair_model =  SequenceTagger.load('kaliani/flair-ner-skill')

2026-05-21 10:51:54,451 SequenceTagger predicts: Dictionary with 7 tags: O, S-SKILL, B-SKILL, E-SKILL, I-SKILL, <START>, <STOP>


Define Filter Mappings

In [51]:
# Map experience level to linkedIn URL code
experience_level_mapping= {
    "Internship": "f_E=1",
    "Entry Level": "f_E=2",
    "Associate": "f_E=3",
    "Mid-Senior Level": "f_E=4"
}

# Map work type to LinkedIn URL code
work_type_mapping={
   "On-site": "f_WT=1",
   "Remote": "f_WT=2",
   "Hybrid": "f_WT=3"
}

# Map time filter to LinkedIn URL codes
time_filter_mapping = {
    "Past 24 hours": "f_TPR=r86400",
    "Past week": "f_TPR=r604800",
    "Past month": "f_TPR=r2592000"
}


In [52]:
description = """Overview

As Data-logger, you will provide technical expertise and support to deliver exceptional service for data capture, management, and reporting throughout the entire lifecycle for wellsite drilling & workover operations. Using data reporting tools and techniques, you will apply your knowledge and understanding to ensure daily reporting of wellsite activity. This includes data capture from various service providers, client organization, real-time data sources and other data formats to ensure streamlined data capture via reporting tools. You will also monitor the reporting for mandatory parameters and maintain records to monitor daily reporting quality. You will receive training that expands your technical knowledge and hands-on skillset.

Responsibilities

Perform daily reporting of wellsite operations and be accountable for overall performance of timely data capture and reporting including configuration, loading, QC, monitoring and troubleshooting of operational activity for the asset / location
Ensure seamless data capture with reporting application including monitoring of mandatory reports, data sources, report formats including units, identifiers, meanings, normal value ranges to identify and fix data issues proactively.
Learn the reporting application, system installation and configuration, troubleshooting process and keep track of issues and resolution
Understand the role of different stakeholders and coordinate project data collection across service companies, client organization, real time data providers
Provide support to the on-site team, collaborate with the project team, client organization and remote operations support to ensure smooth and reliable function of application software and systems, including system upgrades, troubleshooting and maintenance activities
Monitor service quality, understand and utilize the Incident Escalation Procedures and pre-emptively inform supervisors when potential issues arise
Understand and Implement the Data Confidentiality Standards and IT Security Policies across all activities related to data capture, storage, transfer and archival
Maintain HSE Scorecards and actively contribute to HSE Safety Programs

Locations: Mehsana, Ahmedabad, Dehra Doon, East India, Assam etc.

Minimum Qualification And Experience

Minimum 3 years' Diploma in E&T / IT / Electronics / Computers/ EC / Electrical / Mechanical / Petroleum or Graduate in IT/BCA or equivalent/higher.
Minimum 2 years' experience of working in Oil & Gas/drilling company/organization
Good verbal and written communication skills in English.

Minimum Qualification And Experience

Minimum 3 years' Diploma in E&T / IT / Electronics / Computers/ EC / Electrical / Mechanical / Petroleum or Graduate in IT/BCA or equivalent/higher.
Minimum 2 years' experience of working in Oil & Gas upstream/drilling company/ etc.
Good verbal and written communication skills in English
"""

**Skill Detection Function**

In [53]:
# define function to find skill in description

def get_skills(text):
  #turn text into flair model context
  sentense = Sentence(text)
  # use AI to detect skills
  flair_model.predict(sentense)
  # return List of Skills found
  return[entity.text for entity in sentense.get_spans('ner')]

get_skills(description)

['communication skills in', 'English']

**Scrapper Manager Class**

In [54]:
class ScrapperManager:
  def __init__(self):
    self.stop_event = threading.Event()
    self.current_df = pd.DataFrame()
    self.lock = threading.Lock()

  def reset(self):
    self.stop_event.clear()
    self.current_df = pd.DataFrame()

  def add_job(self, job_data):
    with self.lock:
      new_df = pd.DataFrame([job_data])
      self.current_df = pd.concat([self.current_df, new_df], ignore_index=True)

scraper_manager = ScrapperManager()

**Save to CSV Function**

In [55]:
def save_csv(df, filename="jobs"):
    try:
  # make folder for files
     os.makedirs("saved_jobs", exist_ok=True)
     # get default name with Time stamp
     if not filename:
      filename = f"jobs_{int(time.time())}"
      # buld File path
      full_path = f"saved_jobs/{filename}.csv"
      # save table to csv
      df.to_csv(full_path, index=False)
      # confirmed save work
      return f"saved_to {full_path}"
    except Exception as e:
      return f"Save_error: {str(e)}"

Process job function

In [56]:
def process_job(job, work_type, experience_level, position):
    try:
      # find the job title
      title_element = job.find('h3', class_='base-search-card__title')
      #find company name
      company_element = job.find('a', class_='hidden-nested-link')
      # find job location
      loc_element = job.find('span', class_='job-search-card__location')
      # find job link
      link_element = job.find('a', class_='base-card__full-link')
      # check all data exists
      if not all([title_element, company_element, loc_element, link_element]):
        return None
      # clean title text
      title = title_element.text.strip()
      #clean comapny text
      company = company_element.text.strip()
      # clean location text
      location = loc_element.text.strip()
      # clean link text
      link = link_element['href'].split('?')[0]

      # setup web sessions with retries
      session =  requests.Session()

      # set retry rules for error
      retries = RequestHistory(total = 3, backoff_factor = 0.1, status_forcelist = [429,500,502,503,504])

      # apply retries for session
      session.mount('https://',HTTPAdapter(max_retries=retries))


      #default if description fails
      desc = "Description Not Avilable"

      # empty Skills list
      skills = []

      # wait 2-5 seconds to avoid bans
      time.sleep(random.uniform(2,5))

      # fetch Job Page
      response = session.get(
          link,
          headers={
              # random brouser to seems like human
             'User-Agent': random.choice([
                  'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3',
                  'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.5 (KHTML, like Gecko) Version/14.1.1 Safari/605.1.5'
              ]),
              #set language for consistency
              'Accept-Language': 'en-US,en;q=0.5'
          },
          # TimeOut after 10 seconds
          timeout=10
      )
      #pasrse job page HTML
      job_soup = BeautifulSoup(response.text, 'html.parser')

      # list of places to find description
      description_selectors=[
          'div.description__text',
          'div.show-more-less-html__markup',
          'div.core-selection-container__content',
          'section.core-section-container'
      ]
      # try each description spot
      for selector in description_selectors:
        # find description
        description_element = job_soup.select_one(selector)
        if description_element:
          #clear description element
          desc = description_element.text.strip()
          # find skill with AI
          skills = get_skills(desc)
          break

      return {
          "Position": position,
          "Date": datetime.now().strftime("%Y-%m-%d"),
          "Work_type": work_type,
          "Level": experience_level,
          "Company": company,
          "location": location,
          "Link": f"[{link}]({link})",
          "Description": desc,
          "Skills": ", ".join(skills[:5]) if skills else "No Skills Found"
      }

    except Exception as e:
        # log error if page Fails
        print(f"Error processing job: {str(e)}")
        return None

**Scrape job Function**

In [57]:
# define function to scrape job listing
def scrape_jobs(location, position, work_types, exp_levels, time_filter):

    # setup web session
    session = requests.Session()

    # set retry rules
    retries = RequestHistory(
        total=3,
        backoff_factor=0.1,
        status_forcelist=[429, 500, 502, 503, 504]
    )

    # Apply retries
    session.mount('https://', HTTPAdapter(max_retries=retries))

    # loop through work types
    for work_type in work_types:

        # loop through the experience level
        for exp in exp_levels:

            # stop if user say so
            if scraper_manager.stop_event.is_set():
                return

            try:
                # build linkedin search url
                base_url = (
                    f"https://www.linkedin.com/jobs/search/?keywords={position}"
                    f"&location={location}"
                    f"&{time_filter_mapping[time_filter]}"
                    f"&{work_type_mapping[work_type]}"
                    f"&{experience_level_mapping[exp]}"
                    f"&radius=0"
                )

                try:
                    # get search page
                    response = session.get(base_url, headers={'User-Agent': random.choice([...]),'Accept-Language': 'en-US,en;q=0.5'}, timeout=10)

                    # parse HTML
                    soup = BeautifulSoup(response.text, 'html.parser')

                    # find total job count
                    total_jobs = int(
                        soup.find(
                            'span',
                            class_='results-context-header__job-count'
                        ).text.replace(',', '')
                    )

                except:
                    # default to 25 if count fails
                    total_jobs = 25

                # cap to 100 jobs
                total_jobs = min(total_jobs, 100)

                # leap through pages (25 jobs each)
                for start in range(0, total_jobs, 25):

                    # stop if user say so
                    if scraper_manager.stop_event.is_set():
                        return

                    # wait 2-5 seconds
                    time.sleep(random.uniform(2, 5))

                    # add page number to the url
                    url = f"{base_url}&start={start}"

                    try:
                        # get page
                        response = session.get(url, timeout=10)

                        # parse HTML
                        soup = BeautifulSoup(response.text, 'html.parser')

                        # find all job cards
                        jobs = soup.find_all('div', class_='base-card')

                    except Exception as e:
                        # log page error
                        print(f"fail to scrape {start}: {str(e)}")
                        continue

                    # Mix up Job order
                    random.shuffle(jobs)

                    # process jobs
                    for job in jobs:
                        print(job.text.strip())
                    #Process each job
                    for job in jobs:
                      # if user says no
                      if scraper_manager.stop_event.is_set():
                        return

                      # get job details
                      job_data = process_job(job, work_type, exp, position)
                      if job_data:
                        # Add the Table
                        scraper_manager.add_job(job_data)
                        # update app
                        yield

            except Exception as e:
              # Log big error
                print(f"Scraping Error: {str(e)}")

**Run Scraper function**

In [58]:
def run_scraper(cities, states, position, work_types, exp_levels, time_filter):
  scraper_manager.reset()
  cities_list = [c.strip() for c in cities.split(',') if c.strip()]
  states_list = [s.strip() for s in states.split(',') if s.strip()]
  locations = [f"{city}, {state}" for city in cities_list for state in states_list]
  positions = [p.strip().replace(' ','%20') for p in position.split(',') if p.strip()]

  def worker():
    for loc in locations:
      for pos in positions:
        if scraper_manager.stop_event.is_set():
          return
        for _ in scrape_jobs(loc, pos, work_types, exp_levels, time_filter):
          pass

  thread = threading.Thread(target=worker)
  thread.start()

  while thread.is_alive():
    time.sleep(0.5)
    with scraper_manager.lock:
      yield 'Scraping in Progress...', scraper_manager.current_df

  yield 'Scraping Completed' if not scraper_manager.stop_event.is_set() else "Scraping Stopped!", scraper_manager.current_df

**GRADIO INTERFARANCE**

In [59]:
with gr.Blocks() as app:
    gr.Markdown("""<div style='text-align:center;color:#f67dc3;font-size:2em;font-weight:bold;margin:20px 0;padding:10px'>AI Powered LinkedIn Job Scrapper</div>""")

    with gr.Row():
        with gr.Column():
            cities = gr.Textbox(label="Cities (comma-separated)")
            states = gr.Textbox(label="States/Countries (comma-separated)")
            positions_input = gr.Textbox(label="Positions (comma-separated)")
            work_types = gr.CheckboxGroup(list(work_type_mapping.keys()), label="Work Types")
            exp_levels = gr.CheckboxGroup(list(experience_level_mapping.keys()), label="Experience Levels")
            time_filter = gr.Dropdown(list(time_filter_mapping.keys()), label="Time Filter")

            with gr.Row():
                start_btn = gr.Button("Start Scraping", variant="primary")
                stop_btn = gr.Button("Stop Scraping", variant="secondary")

    status = gr.Textbox(label="Status")
    results = gr.DataFrame(
        headers=["Position", "Date", "Work_type", "Level", "Company", "location", "Link", "Description", "Skills"],
        datatype=["str", "str", "str", "str", "str", "str", "str", "markdown", "str"],
        interactive=False
    )

    with gr.Row():
        filename = gr.Textbox(label="Filename (optional)", placeholder="my_jobs")
        save_btn = gr.Button("Save to CSV", variant="secondary")

    save_status = gr.Textbox(label="Save Status")

    start_btn.click(
        run_scraper,
        inputs=[cities, states, positions_input, work_types, exp_levels, time_filter],
        outputs=[status, results]
    )

    stop_btn.click(lambda: scraper_manager.stop_event.set())
    save_btn.click(save_csv, inputs=[results, filename], outputs=save_status)

if __name__ == "__main__":
    app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://330e679862ce3ebf28.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
